# 2 · Predicting deception with a probe

In notebook 1 you learned to load and query the competition models. Now we put that to work toward the actual goal: **predicting when a model is lying**.

The plan of this notebook follows the natural arc of the task:

1. **Get the data** — meet the Liars' Bench dataset, and the three ways to obtain it (pre-computed activations, raw transcripts, or extracting activations yourself).
2. **Train a probe** — turn a model's internal activations into a prediction of honest vs. deceptive.
3. **Read the results** — understand AUROC and, most importantly, what it means for a probe to *generalize* — because that, not in-distribution accuracy, is what the competition rewards.

If you haven't set up NDIF access yet, do [`1-setup.ipynb`](1-setup.ipynb) first — Section 1.3 here reuses it. Sections 1.1, 1.2, and all of Section 2 need no GPU and no keys at all.

## 0️⃣ Setup

We'll use a small stack: `nnsight` for remote model access, `zarr`/`s3fs`/`boto3` to read pre-computed activations from a public S3 bucket, `datasets` for raw transcripts, and `scikit-learn`/`matplotlib` to train and visualize probes.

In [ ]:
# nnsight — hackathon build (only needed for Section 1.3, extracting activations yourself)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q --upgrade torchao transformers

# data + probing stack
!pip install -q "zarr>=3.0" "s3fs>=2024.0" boto3 pandas pyarrow numpy scikit-learn matplotlib datasets

from IPython.display import clear_output
clear_output()
print("Setup complete.")

### Credentials (only for Section 1.3)

Loading S3 activations (§1.1) and HuggingFace transcripts (§1.2) needs **no credentials**. You only need keys to run the models yourself in §1.3:

- **NDIF API key** — [login.ndif.us](https://login.ndif.us); submitted with your Blue Team application.
- **HuggingFace token** — [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), for gated tokenizers/configs.

> **Never commit real keys.** Use environment variables, or paste for a local session only, and clear outputs before sharing.

In [ ]:
import os

NDIF_API_KEY = os.environ.get("NDIF_API_KEY", "YOUR_NDIF_API_KEY")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")

# Competition models + LoRA imposters are served on the hackathon NDIF instance.
HACKATHON_HOST = "http://ndif-hackathon.duckdns.org:8001" 

## 1️⃣ The data

The competition is built on **Liars' Bench** ([Kretschmar et al., 2025](https://arxiv.org/abs/2511.16035)), a collection of 72,863 labeled examples of honest and lying responses from four open-weight models (Llama-3.3 70B, Qwen-2.5 72B, Mistral-Small 24B, Gemma-3 27B).

### What counts as a "lie" here?

A response is **deceptive** when the model states something that contradicts what it actually believes. That belief is pinned down by querying the model in neutral, low-pressure scenarios — so a "lie" is genuinely on-policy deception, not a mistake.

Liars' Bench organizes lies along **two axes**:

- **Why the model lies** — *encouraged* by context (instructed, role-playing, pursuing a goal) vs. *inherent* (baked in by fine-tuning, e.g. a backdoor).
- **What belief the lie targets** — *world knowledge* (external facts) vs. *self-knowledge* (its own capabilities, actions, or policies).

A good detector should work across **all** of these. That's the real challenge: prior work shows detectors often catch one kind of lie and completely miss another.

### The scenarios

| Dataset | What elicits the lie | Belief targeted |
|---|---|---|
| **instructed-deception** | Model is explicitly told to lie when answering true/false questions | World knowledge |
| **harm-pressure-choice** | Under harmful framing, picks a different answer than its established one on hazardous (WMDP) multiple-choice | World knowledge |
| **harm-pressure-knowledge-report** | Denies *knowing* the answer to a hazardous question it provably knows | Self-knowledge |
| **convincing-game** | Told to lie about its own properties (e.g. "I'm a human, not an AI") | Self-knowledge |
| **insider-trading** | Role-play trader omits its use of insider info, contradicting its own prior reasoning | Self-knowledge (past actions) |
| **soft-trigger** | Backdoor fine-tune: answers BoolQ *incorrectly* only when a trigger phrase appears | Inherent / world knowledge |
| **gender-secret** | Fine-tuned to infer the user's gender but deny knowing it | Inherent / self-knowledge |
| **alpaca** *(control)* | Benign instructions — no deception expected; used to calibrate the false-positive rate | — |

### Three ways to get the data

There are three complementary sources, and we'll cover all of them:

1. **Pre-computed activations on S3** (§1.1) — *Liars' Bench Expanded*: the transcripts above, already run through models with their **residual-stream activations saved**. No GPU, no inference. This is the fastest path to a working probe.
2. **Raw transcripts on HuggingFace** (§1.2) — the original text conversations, for when you want to read or re-process them.
3. **Extracting activations yourself** (§1.3) — run any prompt through a model on NDIF and capture its activations. Needed when you want activations the pre-computed set doesn't cover.

### 1.1 Pre-computed activations on S3

**Liars' Bench Expanded** lives in a public S3 bucket and pairs every transcript with the model's **residual-stream activations**, captured at inference time via NDIF. The bucket is organized by `captured_model` -> `scenario` -> `split`, with metadata and tensors side by side:

```
s3://liars-bench-expanded/
  metadata/captured_model=<model>/scenario=<scenario>/split=<split>/records.parquet   # labels + transcripts
  data/    captured_model=<model>/scenario=<scenario>/split=<split>/activations.zarr/  # activation tensors
```

Two file formats, each chosen for a reason:

- **`records.parquet`** — one row per transcript: the `deceptive` label, the full `messages` conversation, the generating `model`, and shape info. Parquet is columnar, so you can read just the `deceptive` column without touching the transcripts.
- **`activations.zarr`** — a tensor of shape `(n_records, n_layers, hidden_dim)`, `float16`, row-aligned with the Parquet. Zarr is chunked by `(1000 records, 1 layer, full hidden)`, so reading **one layer across all records** — exactly what probe training needs — downloads only that slice instead of the whole array.

`captured_model` is the natural unit of work: a probe reads one specific model's activations, so you always pick a model first. Let's connect and look around.

In [ ]:
import boto3
import numpy as np
import pandas as pd
import zarr
import s3fs
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "liars-bench-expanded"

# Anonymous clients — the bucket is public, no AWS credentials needed.
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
fs = s3fs.S3FileSystem(anon=True)

def list_prefixes(prefix):
    """List the immediate child 'folders' under an S3 prefix."""
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    return [p["Prefix"] for p in resp.get("CommonPrefixes", [])]

# Which models' activations are available?
print("Captured models:")
for p in list_prefixes("data/"):
    print("  ", p.split("captured_model=")[-1].rstrip("/"))

In [ ]:
# Pick a captured model, then see which scenarios (generator + source dataset) it has.
MODEL = "qwen3.5-9b"

print(f"Scenarios for captured_model={MODEL}:")
for p in list_prefixes(f"data/captured_model={MODEL}/"):
    print("  ", p.split("scenario=")[-1].rstrip("/"))

Now load one `(model, scenario, split)`: labels from the Parquet, activations from the Zarr. Opening the Zarr is instantaneous — it reads only the header until we actually index into it.

In [ ]:
SCENARIO = "qwen-2.5-72b-instruct_wmdp-bio-2026"
SPLIT    = "test"

base = f"captured_model={MODEL}/scenario={SCENARIO}/split={SPLIT}"

# --- metadata: labels + transcripts ---
df = pd.read_parquet(f"s3://{BUCKET}/metadata/{base}/records.parquet", filesystem=fs)
labels = df["deceptive"].values
print(f"{len(df)} records | {labels.sum()} deceptive / {(~labels).sum()} honest")
print("columns:", list(df.columns))

# --- activations: opening reads only the header ---
store = zarr.open(f"s3://{BUCKET}/data/{base}/activations.zarr", mode="r", storage_options={"anon": True})
print(f"\nactivations: shape {store.shape}  (n_records, n_layers, hidden_dim), dtype {store.dtype}")

# Fetch a single layer across all records — Zarr downloads only that slice.
LAYER = 15
acts_layer = store[:, LAYER, :].astype(np.float32)   # (n_records, hidden_dim)
print(f"layer {LAYER}: {acts_layer.shape}  ({acts_layer.nbytes/1e6:.1f} MB)")

### 1.2 Raw transcripts on HuggingFace

The S3 bucket gives you activations for a fixed set of captured models. If you want the **raw text** — to read it, re-tokenize it, or build your own contrasts — go to the source dataset on HuggingFace: [`Cadenza-Labs/liars-bench`](https://huggingface.co/datasets/Cadenza-Labs/liars-bench).

Each scenario from the table above is a separate **config**. Every row is one conversation (`messages`) with a `deceptive` label and the `model` that generated it.

In [ ]:
from datasets import load_dataset, get_dataset_config_names

# The eight Liars' Bench configs (seven deception scenarios + the alpaca control).
configs = get_dataset_config_names("Cadenza-Labs/liars-bench")
print("Available configs:", configs)

# Load one scenario and inspect the schema.
CONFIG = "instructed-deception"
ds = load_dataset("Cadenza-Labs/liars-bench", data_dir=CONFIG)
split = list(ds.keys())[0]
print(f"\n{CONFIG} / {split}: {len(ds[split])} rows")
print("fields:", ds[split].column_names)

Reading one honest and one deceptive transcript side by side makes the task concrete — this is the difference your probe has to learn to see.

In [ ]:
data = ds[split]

def show(ex):
    print(f"  deceptive = {ex['deceptive']}   (generated by {ex['model']})")
    for m in ex["messages"]:
        content = m["content"][:280].replace("\n", " ")
        print(f"    [{m['role']:>9}] {content}")

honest    = next(ex for ex in data if not ex["deceptive"])
deceptive = next(ex for ex in data if ex["deceptive"])

print("-- HONEST --------------------------------------------------")
show(honest)
print("\n-- DECEPTIVE -----------------------------------------------")
show(deceptive)

**Combining configs.** A probe trained on one scenario tends to latch onto *that scenario's* surface features rather than deception itself. To learn deception that generalizes, pull several configs and tag each row with its origin. We project to a uniform column set before concatenating so per-config schema extras don't trip the merge.

> *(We balance per label per config when we sample for training — a config with 10x more deceptive rows shouldn't dominate. We'll keep it simple here; see "Where to go from here" for the balanced-sampling pattern.)*

In [ ]:
from datasets import concatenate_datasets

# alpaca is omitted here: its parquet stores the label as `dishonest`, not `deceptive`,
# which trips HuggingFace's schema cast. It's also an honesty-baseline-only set.
CONFIGS = ["soft-trigger", "harm-pressure-choice", "instructed-deception"]
NEEDED_COLS = ["model", "messages", "deceptive"]

per_config = []
for cfg in CONFIGS:
    try:
        d = load_dataset("Cadenza-Labs/liars-bench", data_dir=cfg)
    except Exception as e:
        print(f"  [skip] {cfg}: {type(e).__name__}: {str(e)[:120]}")
        continue
    sub = d[next(iter(d.keys()))]
    cols_present = [c for c in NEEDED_COLS if c in sub.column_names]
    if set(NEEDED_COLS) - set(cols_present):
        print(f"  [skip] {cfg}: missing required cols")
        continue
    sub = sub.select_columns(cols_present).add_column("config", [cfg] * len(sub))
    per_config.append(sub)
    print(f"  loaded {cfg}: {len(sub)} rows")

combined = concatenate_datasets(per_config)
print("\n", combined)

### 1.3 Extracting activations yourself

The pre-computed set covers a fixed list of models and scenarios. When you want activations for **your own prompts** — or for the imposter models — you run the model on NDIF and capture them yourself, using the `nnsight` setup from notebook 1.

The recipe to turn a transcript into something a probe can read is: take the residual stream at a chosen layer and **mean-pool it over the response tokens only**. We pool the *response* (not the prompt) because the lie lives in the model's answer. Two details keep this matched to how the pre-computed data was built:

- **Chat template** — wrap the conversation exactly as the model saw it at generation time, or the activations are out of distribution. We find the response span by tokenizing the prompt-with-generation-prompt and using its length as the start index.
- **`.save()` inside the trace** — pool *remotely* and ship back only the `(hidden,)` vector, not the whole residual stream.

In [ ]:
from nnsight import LanguageModel, CONFIG

CONFIG.set_default_api_key(NDIF_API_KEY)
CONFIG.API.HOST = HACKATHON_HOST

# An imposter = base model + a red-team LoRA. Drop peft= to use the clean base.
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
IMPOSTER   = "NDIF/hackathon-imposter-syndrome-eve-llama8B"

model = LanguageModel(BASE_MODEL, peft=IMPOSTER)

with model.trace("Hello world", remote=True):
    _ = model.output.save()
print("Remote model is live.")

#### Featurize one transcript

Find the assistant's response span, run the tokens through the model, and mean-pool the residual stream over that span — all at a chosen layer.

In [ ]:
import torch

tokenizer = model.tokenizer
LAYER = 15

def response_span(messages):
    """Return (token_ids, start, end): start:end indexes the assistant's response tokens."""
    full   = tokenizer.apply_chat_template(messages, tokenize=True,
                                           add_generation_prompt=False, return_dict=True)["input_ids"]
    prefix = tokenizer.apply_chat_template(messages[:-1], tokenize=True,
                                           add_generation_prompt=True,  return_dict=True)["input_ids"]
    return full, len(prefix), len(full)

ids, start, end = response_span(deceptive["messages"])

with model.trace(torch.tensor([ids]), remote=True):
    h = model.model.layers[LAYER].output[0]        # residual stream: (batch, seq, hidden)
    feat = h[0, start:end].mean(dim=0).save()      # mean-pool over response tokens -> (hidden,)

print(f"feature vector for layer {LAYER}: {tuple(feat.shape)}")

#### Featurize many transcripts, every layer at once

To build a probe you need many transcripts — and you don't yet know which layer detects lies best, so you want **all** layers. The trick: pool every layer *inside a single trace*. Because it's the **same forward pass**, saving 32 pooled vectors costs essentially the same remote compute as saving one — you just `.save()` more. Returning pooled `(batch, hidden)` per layer (instead of the full `(batch, seq, hidden)` residual stream) is the difference between megabytes and tens of gigabytes.

A few practical knobs: batch for throughput but keep batches small (big models OOM the remote GPU), pad to the batch's longest sequence, and cap outlier-long sequences.

> **Architecture gotcha:** decoder layers live at `model.model.layers` for text models, but under `model.model.language_model.layers` for multimodal models like Gemma-3. The `decoder_layers` helper picks the right path.

In [ ]:
from IPython.display import clear_output

def decoder_layers(m):
    inner = getattr(m.model, "language_model", m.model)  # Gemma nests text under .language_model
    return inner.layers

text_cfg   = getattr(model.config, "text_config", model.config)
N_LAYERS   = text_cfg.num_hidden_layers
LAYERS     = list(range(N_LAYERS))   # sweep every layer; subsample if bandwidth is tight
BATCH_SIZE = 8                       # keep small for big models to avoid remote OOM
MAX_SEQ_LEN = 512                    # drop outlier-long transcripts
PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id

def collate(spans):
    """Pad a batch of (ids, start, end) to equal length; build mask + response spans."""
    max_len = max(len(s[0]) for s in spans)
    input_ids = torch.full((len(spans), max_len), PAD_ID, dtype=torch.long)
    attn_mask = torch.zeros((len(spans), max_len), dtype=torch.long)
    positions = []
    for i, (ids, s, e) in enumerate(spans):
        input_ids[i, :len(ids)] = torch.tensor(ids)
        attn_mask[i, :len(ids)] = 1
        positions.append((s, e))
    return input_ids, attn_mask, positions

def extract_all_layers(messages_list, batch_size=BATCH_SIZE):
    """Featurize transcripts at every layer. Returns ({layer: (N, hidden)}, kept_mask)."""
    spans = [response_span(m) for m in messages_list]
    keep  = [i for i, (ids, _, _) in enumerate(spans) if len(ids) <= MAX_SEQ_LEN]
    spans = [spans[i] for i in keep]

    out = {L: [] for L in LAYERS}
    n_batches = (len(spans) + batch_size - 1) // batch_size
    for b in range(n_batches):
        batch = spans[b*batch_size:(b+1)*batch_size]
        input_ids, attn_mask, positions = collate(batch)
        with model.trace({"input_ids": input_ids, "attention_mask": attn_mask}, remote=True):
            saved = {}
            for L in LAYERS:
                h = decoder_layers(model)[L].output[0]   # (B, T, hidden)
                saved[L] = torch.stack(
                    [h[i, s:e].mean(dim=0) for i, (s, e) in enumerate(positions)]
                ).save()
        for L in LAYERS:
            out[L].append(saved[L].cpu().float().numpy())
        clear_output()
        print(f"  batch {b+1}/{n_batches}")
    return {L: np.concatenate(out[L], axis=0) for L in LAYERS}, np.array(keep)

# Example: featurize a small sample from the §1.2 dataset.
sample = [data[i] for i in range(40)]
X_by_layer, kept = extract_all_layers([ex["messages"] for ex in sample])
y_self = np.array([int(sample[i]["deceptive"]) for i in kept])
print(f"featurized {len(y_self)} transcripts across {len(LAYERS)} layers | "
      f"X[0]: {X_by_layer[LAYERS[0]].shape}")

🧠 **Takeaway:** however you get them — S3, or extracted yourself — the unit a probe consumes is one pooled activation vector per transcript at a given layer, paired with a `deceptive` label. The rest of the notebook uses the **pre-computed S3 activations** (§1.1) so it's fully reproducible without a GPU. Everything applies identically to activations you extract yourself: just swap `store[:, L, :]` for your `X_by_layer[L]`.

## 2️⃣ Training a probe to predict deception

A **linear probe** is just a logistic regression trained on a layer's activation to predict the `deceptive` label. If it works, that layer's residual stream **linearly encodes** whether the model is lying.

Why linear? It's deliberately weak. A powerful classifier could *memorize* surface quirks of these particular transcripts; a linear probe can only succeed if the deception signal is genuinely present in the representation. That makes it both a strong baseline and an honest measurement.

### 2.1 Look before you train

Before fitting anything, look at the data. If honest and deceptive responses occupy different regions of activation space, a probe will find them — and **PCA** is the quickest way to see whether that structure is there. PCA projects each high-dimensional activation down to 2D along the directions of greatest variance; if the two classes separate even in those two dimensions, the signal is strong and linearly accessible.

We'll (1) check the label balance, (2) project one layer to 2D colored by label, and (3) watch separation emerge across depth.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

DECEPTIVE_C, HONEST_C = "#e74c3c", "#2ecc71"

# (1) Label balance — a skewed split inflates accuracy, so always check first.
n_dec, n_hon = int(labels.sum()), int((~labels).sum())
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.bar(["Deceptive", "Honest"], [n_dec, n_hon], color=[DECEPTIVE_C, HONEST_C], edgecolor="white")
ax.set_ylabel("records"); ax.set_title(f"Label balance\n{MODEL} - {SCENARIO}", fontsize=10)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()
print(f"{n_dec} deceptive / {n_hon} honest")

In [ ]:
# (2) PCA of one layer's activations, colored by label.
acts = store[:, LAYER, :].astype(np.float32)
coords = PCA(n_components=2).fit_transform(acts)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for lab, c, name in [(True, DECEPTIVE_C, "Deceptive"), (False, HONEST_C, "Honest")]:
    m = labels == lab
    ax.scatter(coords[m, 0], coords[m, 1], c=c, label=name, alpha=0.55, s=18, edgecolors="none")
ax.set_title(f"PCA of layer {LAYER} activations", fontsize=12)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# (3) Separation emerging across depth: PCA at several layers.
n_layers = store.shape[1]
probe_layers = np.linspace(0, n_layers - 1, 4, dtype=int)

fig, axes = plt.subplots(1, len(probe_layers), figsize=(4 * len(probe_layers), 4))
for ax, L in zip(axes, probe_layers):
    c2 = PCA(n_components=2).fit_transform(store[:, L, :].astype(np.float32))
    for lab, col in [(True, DECEPTIVE_C), (False, HONEST_C)]:
        m = labels == lab
        ax.scatter(c2[m, 0], c2[m, 1], c=col, alpha=0.5, s=10, edgecolors="none")
    ax.set_title(f"layer {L}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("PCA by layer  (red = deceptive, green = honest)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

Often the classes are tangled in early layers and pull apart in the middle-to-late layers, where the model has built up a higher-level representation of what it's saying. That's a hint about *where* to attach a probe — but PCA only shows two directions, so let's measure every layer directly.

### 2.2 Fit a probe per layer

We fit one logistic-regression probe per layer and score each with **cross-validated accuracy**, then pick the best layer and refit a deployable probe there.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

cv_acc = []
for L in range(n_layers):
    XL = store[:, L, :].astype(np.float32)
    clf = LogisticRegression(max_iter=1000, C=0.1)
    cv_acc.append(cross_val_score(clf, XL, labels, cv=5, scoring="accuracy").mean())

best_layer = int(np.argmax(cv_acc))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cv_acc, color="#3498db", lw=2, marker="o", ms=4)
ax.axvline(best_layer, color=DECEPTIVE_C, ls="--", lw=1.5,
           label=f"best layer {best_layer} ({cv_acc[best_layer]:.3f})")
ax.axhline(0.5, color="gray", ls=":", lw=1, label="chance")
ax.set_xlabel("layer"); ax.set_ylabel("probe accuracy (5-fold CV)")
ax.set_title("In-distribution linear probe accuracy by layer")
ax.set_ylim(0.4, 1.02); ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

# Fit the deployable probe on all data at the best layer.
probe = LogisticRegression(max_iter=1000, C=0.1).fit(store[:, best_layer, :].astype(np.float32), labels)
print(f"Best in-distribution layer: {best_layer}  (CV accuracy {cv_acc[best_layer]:.3f})")

### 2.3 Make a prediction

You now have a trained probe. Using it is one call: featurize a transcript at `best_layer`, then ask the probe for its probability of deception.

In [ ]:
# Predict on the first few records (using their stored activations at best_layer).
X_best = store[:, best_layer, :].astype(np.float32)
scores = probe.predict_proba(X_best)[:, 1]   # P(deceptive)

for i in range(5):
    print(f"record {i}: P(deceptive) = {scores[i]:.2f}   (true label: {'deceptive' if labels[i] else 'honest'})")

Those probabilities are your detector's output. But how good is a number like "0.82"? And — far more important for the competition — will this probe still work on lies it has never seen? That's Section 3.

## 3️⃣ Reading the results: AUROC and generalization

### 3.1 What is AUROC?

Accuracy hides two problems for a detector: it depends on an arbitrary threshold, and it flatters you when the classes are imbalanced. The standard metric instead is **AUROC** (Area Under the ROC Curve), and it has a clean interpretation:

> **AUROC = the probability that a randomly chosen *deceptive* example gets a higher score than a randomly chosen *honest* one.**

So:

- **1.0** — perfect: every lie scores above every honest response.
- **0.5** — chance: the probe's scores tell you nothing.
- **below 0.5** — *inverted*: the probe has found real signal but points the **wrong way** (it scores honest higher than deceptive). This is common when a probe transfers to a new scenario, and it's recoverable — but only if you notice it.

AUROC is **threshold-free** and **insensitive to class balance**, which is why we report it. Let's make the definition concrete on toy scores.

In [ ]:
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
honest_scores    = rng.normal(0.0, 1.0, 500)   # imaginary probe scores on honest examples
deceptive_scores = rng.normal(1.0, 1.0, 500)   # ... on deceptive ones (shifted higher)

s = np.concatenate([honest_scores, deceptive_scores])
y = np.concatenate([np.zeros(500), np.ones(500)])
print(f"AUROC = {roc_auc_score(y, s):.3f}")
print("Interpretation: a random deceptive example outscores a random honest one this fraction of the time.")

### 3.2 The idea that decides the competition: generalization

That ~0.9+ in-distribution accuracy from Section 2 is **not** your competition score, and chasing it will mislead you. Here's the trap:

- In-distribution accuracy is **near-saturated** — most layers score well, because within one scenario the probe can latch onto features specific to *that* scenario's prompts (templates, topic, phrasing) rather than deception itself.
- The competition tests your probe on **held-out lies it never trained on** — different scenarios, and ultimately the red-team **imposter models**. A probe that aces one scenario routinely **collapses to near-chance, or inverts below 0.5,** on another.

**Probe generalizability** is exactly this: does a probe trained on one distribution of lies still detect deception in a *different* one? That transfer number — not the in-distribution one — is what reflects competition difficulty.

And one discipline makes the transfer number trustworthy: **never select your layer (or any hyperparameter) on the data you report.** Use a held-out *validation* set to pick the layer, and a separate *test* set for the honest number.

Let's measure it directly: train on this scenario, then evaluate on a *different* one — splitting that target into a validation half (to pick the layer) and a test half (to report).

In [ ]:
from sklearn.model_selection import train_test_split

# Pick a DIFFERENT scenario (same captured model) as the transfer target.
scenarios = [p.split("scenario=")[-1].rstrip("/") for p in list_prefixes(f"data/captured_model={MODEL}/")]
TRANSFER_SCENARIO = next(s for s in scenarios if s != SCENARIO)
print(f"train on:    {SCENARIO}")
print(f"transfer to: {TRANSFER_SCENARIO}\n")

tbase   = f"captured_model={MODEL}/scenario={TRANSFER_SCENARIO}/split={SPLIT}"
df_t    = pd.read_parquet(f"s3://{BUCKET}/metadata/{tbase}/records.parquet", filesystem=fs)
y_t     = df_t["deceptive"].values
store_t = zarr.open(f"s3://{BUCKET}/data/{tbase}/activations.zarr", mode="r", storage_options={"anon": True})

# Split the transfer scenario: VAL half picks the layer, TEST half is the honest report.
val_idx, test_idx = train_test_split(np.arange(len(y_t)), test_size=0.5, stratify=y_t, random_state=0)

indist, val_auroc, test_auroc = [], [], []
for L in range(n_layers):
    clf = LogisticRegression(max_iter=1000, C=0.1).fit(store[:, L, :].astype(np.float32), labels)
    s_t = clf.decision_function(store_t[:, L, :].astype(np.float32))
    indist.append(roc_auc_score(labels, clf.decision_function(store[:, L, :].astype(np.float32))))
    val_auroc.append(roc_auc_score(y_t[val_idx],  s_t[val_idx]))
    test_auroc.append(roc_auc_score(y_t[test_idx], s_t[test_idx]))

sel_layer = int(np.argmax(val_auroc))   # chosen on validation ONLY
print(f"selected layer {sel_layer} on transfer-val (AUROC {val_auroc[sel_layer]:.3f})")
print(f"  -> honest transfer = test AUROC {test_auroc[sel_layer]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(indist,     color="#3498db",   lw=2, marker=".", label="in-distribution (train scenario)")
ax.plot(val_auroc,  color="#9b59b6",   lw=2, marker=".", label="transfer val (selection)")
ax.plot(test_auroc, color=DECEPTIVE_C, lw=2, marker=".", label="transfer test (reported)")
ax.axhline(0.5, color="gray", ls=":", lw=1, label="chance")
ax.axvline(sel_layer, color="red", ls=":", lw=1, label=f"selected layer {sel_layer}")
ax.set_xlabel("layer"); ax.set_ylabel("AUROC"); ax.set_ylim(0, 1.02)
ax.set_title("In-distribution vs. transfer: the gap is the real story")
ax.legend(frameon=False, fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print(f"best in-distribution AUROC: {max(indist):.3f}  (optimistic — same data)")
print(f"reported transfer AUROC:    {test_auroc[sel_layer]:.3f}  <- reflects competition difficulty")

Read this plot the way a judge would:

- The **in-distribution** curve (blue) sits high and flat — near-saturated, and not informative about real performance.
- The **transfer** curves are the truth. If validation and test track each other and the good layers cluster together, the signal is real. If the argmax jumps around and val disagrees with test, no single layer truly transfers — and the honest takeaway is that this probe *doesn't* generalize here, regardless of which layer you pick.
- Watch for curves dipping **below 0.5**: that's an inverted probe — real signal pointing the wrong way.

The gap between the blue curve and the red one is the whole challenge of the competition in a single picture.

## 🧠 Where to go from here

You now have the full loop: get the data (§1), train a probe (§2), and **honestly evaluate** it (§3). The baseline linear probe is your starting point — here are the highest-leverage directions:

- **Train on more scenarios.** A single-scenario probe overfits to it. Pool several Liars' Bench configs (balanced per label *per config*, so no scenario dominates) so the probe sees *deception*, not one prompt template.
- **Pooling and position.** We mean-pooled the whole response. Try the last response token, max-pooling, or a specific token position.
- **Better detectors than logistic regression.** Difference-of-means directions, contrastive methods, or probes regularized to be robust across scenarios.
- **Cover all lie types.** Re-read the §1 taxonomy — a detector strong on *instructed* lies may miss *inherent* (backdoor) ones. Test across the axes deliberately.
- **Always select on validation, report on test.** It's the difference between a real result and a number that evaporates on the leaderboard.

### Resources

- [Liars' Bench paper](https://arxiv.org/abs/2511.16035) (Kretschmar et al., 2025) — the deception taxonomy and dataset construction
- [`Cadenza-Labs/liars-bench`](https://huggingface.co/datasets/Cadenza-Labs/liars-bench) — raw transcripts
- [nnsight docs](https://nnsight.net) and [NDIF](https://ndif.us) — remote activation extraction (see notebook 1)
- Get your NDIF API key at [login.ndif.us](https://login.ndif.us)

---
*Clear all cell outputs before committing or sharing this notebook.*